In [1]:
import sys
sys.path.append('../src/')
from autocorrelations import *

# G2 Calculation

In [ ]:
#Calculate the G2 autocorrelations for the entire image
mask = np.ones(det_corr.shape[1:])

#Delays per level. Increase for more sampling
dpl = 10

#IF are intensities future, IP are intensities past. Used to normalize g2
#I am not sure what the mask here does, may not need it
sumI,G2,IF,IP,_ = g2calc(det_corr[:,:,:],mask,dpl)

# Calculate Frame Spacing
nframes = det.shape[0]

#Delay times. Plot your g2 against this.
delay = frameSpacing * finddelays(nframes,dpl,1)

# Create some loop over your masks to normalize the g2s

In [ ]:
#Results Arrays
G2_result = np.zeros((masks.shape[0],delay.shape[1]))
G2_error = np.zeros((masks.shape[0],delay.shape[1]))


for mm,m in enumerate(masks):
    
    # Last bit of filtering 
    G2_masked = G2[:,m==1]

    # Removes the detector borders
    hot_pixel_mask = G2_masked[0,:] < 10**(18) 
    
    # This address an issue I sometimes (rarely) encounter with shape mismatches. 
    # No idea why it happens but this seems to fix it with no noticeable artifact.
    if np.abs(G2_masked.shape[0]-delay.shape[1])==0:
        last_index = G2_masked.shape[0]
    if np.abs(G2_masked.shape[0]-delay.shape[1])>0:
        last_index = -np.abs(G2_masked.shape[0]-delay.shape[1])

    # Initialize output arrays
    g2dummy = np.zeros_like(G2_masked)

    # Compute partition normalization factor (avoiding division by zero)
    part_norm = np.mean(IF[:, m == 1][:, hot_pixel_mask == 1],axis=1)*np.mean(IP[:, m == 1][:, hot_pixel_mask == 1], axis=1)

    # Normalize G2
    g2dummy = np.divide(G2_masked, part_norm[:, np.newaxis], out=np.full_like(G2_masked, np.nan), where=part_norm[:, np.newaxis] != 0)

    # Compute mean and standard error over the mask
    valid_pixels = hot_pixel_mask == 1

    #Calculate mean and error for the current mask
    G2_result[mm, :] = np.mean(g2dummy[:, valid_pixels][:last_index], axis=1)
    G2_error[mm, :] = np.std(g2dummy[:, valid_pixels][:last_index], axis=1) / np.sqrt(np.sum(valid_pixels))
    

# Two Time

## Plot Mask you want to calculate two-time for

In [ ]:
M = 0

# Plot ROI
fig, ax = plt.subplots()

# Overlay the second image with a 'plasma' color scale and some transparency
ax.imshow(np.sum(det[0:1,:,:],axis=0)*masks[M,:,:]+1,cmap='nipy_spectral',norm=LogNorm(vmin=1, vmax=500))
#ax.imshow((det[:,:,:]*masks[M,:,:])[:,:,det.shape[2]//2].T+1,cmap='nipy_spectral',norm=LogNorm(vmin=1, vmax=500))
ax.set_title(f'Ring {M}')
# Display the plot
plt.show()

## Calculate Two-Time

In [ ]:
#Pick Area for mask only
IMG_temp = det[:,masks[M,:,:]==1]
#Reshapes so it's still 3-dimensional
IMGS_NORM = IMG_temp.reshape((IMG_temp.shape[0],IMG_temp.shape[1],1))

C = twotime(IMGS_NORM)

#Extent for plotting
C_extent = [0,frameSpacing*det_corr.shape[0],0,frameSpacing*det_corr.shape[0]]


## Plot Two-Time

In [ ]:
import matplotlib.pyplot as plt
%matplotlib notebook

#C/np.size(det[:,masks[M,:,:]==1])
fig,ax = plt.subplots()
ax.set_ylabel('t2 (s)')
ax.set_xlabel('t1 (s)')
ax.set_title(f'Ring {M}')
#ax.set_title('Whole Mask')
im=ax.imshow(C,origin="lower",extent=C_extent,cmap='plasma')
fig.colorbar(im, ax=ax)

plt.show